In [ ]:
import pandas as pd
import spacy
from itertools import combinations
from collections import Counter
import networkx as nx
import re

# --- 1. CONFIGURAÇÃO DO GITHUB ---
USUARIO_GIT = "OtavioAntonio"
REPOSITORIO = "DCA3702_analise-ner-videocasts"
BRANCH = "main"
PASTA = "data"  # No seu caso, os arquivos estão dentro da pasta 'data'

# URL Base correta para arquivos RAW (brutos)
URL_BASE = f"https://raw.githubusercontent.com/{USUARIO_GIT}/{REPOSITORIO}/{BRANCH}/{PASTA}/"

meus_arquivos = [
    "u321m25rKXc.csv", "Rz-4ulRKnz4.csv", "8NLzc9kobDk.csv",
    "Q8Qk_3a3lUw.csv", "qCbfTN-caFI.csv", "QJtPROVsePk.csv",
    "2oxdDKHdcM8.csv", "_El9riy9Zjw.csv", "0cn3VBjfN8g.csv"
]

COLUNA_TEXTO = 0
CATEGORIAS_ALVO = ['PERSON', "ORG", "GPE"]
MIN_PESO_GEPHI = 5
MODO_SEGMENTACAO = 'sentenca'
K_VALOR = 250

# --- 2. PREPARAÇÃO DO MODELO ---
try:
    nlp = spacy.load("en_core_web_md")
except:
    import os
    os.system("python -m spacy download en_core_web_md")
    nlp = spacy.load("en_core_web_md")

def segmentar_texto(texto_completo, modo, k=500):
    if modo == 'paragrafo':
        return [p.strip() for p in re.split(r'\n+', texto_completo) if len(p.strip()) > 10]
    elif modo == 'sentenca':
        doc_temp = nlp(texto_completo)
        return [sent.text.strip() for sent in doc_temp.sents]
    elif modo == 'k-caracteres':
        return [texto_completo[i:i+k] for i in range(0, len(texto_completo), k)]
    return [texto_completo]

# --- 3. PROCESSAMENTO (AQUI ONDE OS DADOS SÃO BAIXADOS) ---
todos_os_pares = []
dicionario_tipos = {}

for arquivo in meus_arquivos:
    try:
        # CONSTRUINDO A URL DO ARQUIVO
        url_final = URL_BASE + arquivo if URL_BASE.endswith("/") else f"{URL_BASE}/{arquivo}"

        print(f"Lendo direto do GitHub: {url_final}")
        df = pd.read_csv(url_final)

        # Unifica o texto
        texto_bruto = " ".join(df.iloc[:, COLUNA_TEXTO].astype(str).tolist())
        unidades_de_texto = segmentar_texto(texto_bruto, MODO_SEGMENTACAO, K_VALOR)

        for doc in nlp.pipe(unidades_de_texto, batch_size=20):
            entidades_na_unidade = []
            for ent in doc.ents:
                if ent.label_ in CATEGORIAS_ALVO:
                    nome_limpo = ent.text.replace("\n", " ").strip()
                    entidades_na_unidade.append(nome_limpo)
                    dicionario_tipos[nome_limpo] = ent.label_

            entidades_unicas = sorted(list(set(entidades_na_unidade)))
            if len(entidades_unicas) > 1:
                todos_os_pares.extend(list(combinations(entidades_unicas, 2)))

        print(f"✔ {arquivo} concluído com sucesso.")

    except Exception as e:
        print(f"❌ Erro ao processar {arquivo}: {e}")

# --- 4. GRAFO E EXPORTAÇÃO ---
contagem_conexoes = Counter(todos_os_pares)
G = nx.Graph()

for (u, v), peso in contagem_conexoes.items():
    if peso >= MIN_PESO_GEPHI:
        G.add_edge(u, v, weight=peso)
        G.nodes[u]['type'] = dicionario_tipos.get(u, "Unknown")
        G.nodes[v]['type'] = dicionario_tipos.get(v, "Unknown")

nome_saida = f"grafo_{MODO_SEGMENTACAO}.gexf"
nx.write_gexf(G, nome_saida)
print(f"\n🚀 Pronto! Grafo gerado: {G.number_of_nodes()} nós e {G.number_of_edges()} arestas.")
print(f"Arquivo salvo como: {nome_saida}")

Lendo direto do GitHub: https://raw.githubusercontent.com/OtavioAntonio/DCA3702_analise-ner-videocasts/main/data/u321m25rKXc.csv
✔ u321m25rKXc.csv concluído com sucesso.
Lendo direto do GitHub: https://raw.githubusercontent.com/OtavioAntonio/DCA3702_analise-ner-videocasts/main/data/Rz-4ulRKnz4.csv
✔ Rz-4ulRKnz4.csv concluído com sucesso.
Lendo direto do GitHub: https://raw.githubusercontent.com/OtavioAntonio/DCA3702_analise-ner-videocasts/main/data/8NLzc9kobDk.csv
✔ 8NLzc9kobDk.csv concluído com sucesso.
Lendo direto do GitHub: https://raw.githubusercontent.com/OtavioAntonio/DCA3702_analise-ner-videocasts/main/data/Q8Qk_3a3lUw.csv
